# 🎯 SIRCCD - Entrenamiento v5: YOLO11l optimizado para H100

## Objetivo
Esta versión está ajustada para **detección** con tus datasets actuales en formato bounding box, buscando un equilibrio más realista entre:
- **precisión alta** para baches y grietas,
- **mejor throughput** en H100,
- y **menor costo** por experimento.

## Idea principal
La versión anterior corría bien, pero no era la más eficiente en costo porque:
- usaba parámetros que `optimizer='auto'` ignoraba,
- generaba más sobrecarga de la necesaria (`plots=True`, checkpoints frecuentes),
- y no dejaba claro cuándo usar una corrida rápida vs una corrida final.

## Perfiles incluidos

| Perfil | Uso | Modelo | ImgSz | Batch | Workers | Plots | Deterministic |
|--------|-----|--------|------:|------:|--------:|------:|--------------:|
| **quick** | pruebas rápidas y tuning barato | YOLO11l | 1024 | 20 | 12 | No | No |
| **balanced** ⭐ | mejor balance precisión/costo | YOLO11l | 1280 | 16 | 12 | No | No |
| **final** | corrida final reproducible | YOLO11l | 1280 | 16 | 8 | Sí | Sí |

## Recomendación
- Usa **`TRAIN_PROFILE = 'quick'`** para buscar hiperparámetros o validar cambios.
- Usa **`TRAIN_PROFILE = 'balanced'`** como modo por defecto en la H100.
- Usa **`TRAIN_PROFILE = 'final'`** solo cuando ya estés seguro de la configuración.

## Nota importante
Esta libreta sigue siendo para **detección**, no segmentación.  
Como tus datasets actuales están anotados con cajas, esta es la ruta correcta por ahora.


---
## 🔧 1. Setup y GPU

In [14]:
!pip install -q ultralytics>=8.4.0

import ultralytics
import torch
import os

print(f"✅ Ultralytics: {ultralytics.__version__}")
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.version.cuda}")

# Info GPU
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\n🖥️ GPU: {gpu_name}")
print(f"   VRAM: {vram_gb:.1f} GB")

# Recomendación automática
is_h100 = 'H100' in gpu_name

if vram_gb >= 70:
    if is_h100:
        rec_batch = 24
        rec_label = 'YOLO11l @ 1280 + batch=24 (H100) 🚀'
    else:
        rec_batch = 18
        rec_label = 'YOLO11l @ 1280 + batch=18 (A100 80GB) 🚀'
elif vram_gb >= 35:
    rec_batch = 14
    rec_label = 'YOLO11l @ 1280 + batch=14 (A100 40GB)'
elif vram_gb >= 20:
    rec_batch = 8
    rec_label = 'YOLO11l @ 1280 + batch=8 (L4)'
elif vram_gb >= 14:
    rec_batch = 4
    rec_label = 'YOLO11l @ 1280 + batch=4 (T4)'
else:
    rec_batch = 2
    rec_label = 'YOLO11l @ 1280 + batch=2 (GPU pequeña)'

print(f"\n💡 Recomendación: {rec_label}")

if is_h100:
    print(f"\n🚀 H100 detectada:")
    print(f"   • 2x más rápido vs A100")
    print(f"   • Batch óptimo: 18-24")

# Montar Drive
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("\n✅ Drive montado")
else:
    print("\n✅ Drive ya montado")

✅ Ultralytics: 8.4.21
✅ PyTorch: 2.10.0+cu128
✅ CUDA: 12.8

🖥️ GPU: NVIDIA H100 80GB HBM3
   VRAM: 79.2 GB

💡 Recomendación: YOLO11l @ 1280 + batch=24 (H100) 🚀

🚀 H100 detectada:
   • 2x más rápido vs A100
   • Batch óptimo: 18-24

✅ Drive ya montado


---
## 📦 2. Extraer Dataset

In [15]:
import zipfile
import glob
from tqdm import tqdm
import os

# Ruta corregida: Directamente en MyDrive
DATASET_PATH = '/content/drive/MyDrive/sirccd_dataset_v1.0.0.zip'
EXTRACT_DIR = '/content/sirccd_dataset'

if os.path.exists(f'{EXTRACT_DIR}/data.yaml'):
    print("✅ Dataset ya extraído")
else:
    if not os.path.exists(DATASET_PATH):
        # Intentar buscar en la carpeta por si acaso
        DATASET_PATH_ALT = '/content/drive/MyDrive/SIRCCD_Dataset/sirccd_dataset_v1.0.0.zip'
        if os.path.exists(DATASET_PATH_ALT):
             DATASET_PATH = DATASET_PATH_ALT
             print(f"⚠️ Encontrado en subcarpeta: {DATASET_PATH}")
        else:
             print(f"❌ ERROR: No se encuentra el archivo en: {DATASET_PATH}")
             print(f"   Verifica que el nombre sea exacto y esté en 'My Drive'.")

    if os.path.exists(DATASET_PATH):
        print(f"📦 Extrayendo dataset desde: {DATASET_PATH}...")
        with zipfile.ZipFile(DATASET_PATH, 'r') as zip_ref:
            for file in tqdm(zip_ref.namelist(), desc="Extrayendo"):
                zip_ref.extract(file, '/content/')
        print("✅ Extraído")

# Conteo
print("\n📊 Dataset:")
total = 0
for split in ['train', 'val', 'test']:
    imgs = len(glob.glob(f'{EXTRACT_DIR}/images/{split}/*.jpg'))
    lbls = len(glob.glob(f'{EXTRACT_DIR}/labels/{split}/*.txt'))
    total += imgs
    print(f"   {split}: {imgs:,} imgs, {lbls:,} labels")
print(f"   TOTAL: {total:,} imágenes")

✅ Dataset ya extraído

📊 Dataset:
   train: 40,330 imgs, 40,543 labels
   val: 11,513 imgs, 11,614 labels
   test: 5,766 imgs, 5,819 labels
   TOTAL: 57,609 imágenes


In [16]:
# === DEDUPLICACIÓN - OMITIDA ===
#
# v1 tuvo MEJOR mAP sin deduplicación (mAP50=0.795 vs 0.78 con dedup)
# Imágenes "similares" actúan como augmentation natural.
# Variaciones sutiles ayudan a la generalización.

print("⏭️ Deduplicación omitida (mejor convergencia sin ella según v1)")

⏭️ Deduplicación omitida (mejor convergencia sin ella según v1)


In [17]:
import yaml

data_config = {
    'path': EXTRACT_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 2,
    'names': {0: 'bache', 1: 'grieta'}
}

with open(f'{EXTRACT_DIR}/data.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("✅ data.yaml listo (2 clases: bache, grieta)")

✅ data.yaml listo (2 clases: bache, grieta)


---
## 🧠 3. Seleccionar Modelo

Elige UNA opción descomentando la línea correspondiente.
Los pesos COCO se descargan automáticamente.

In [18]:
from ultralytics import YOLO
from datetime import datetime

# ╔════════════════════════════════════════════════════════════╗
# ║  MODELO BASE                                               ║
# ╚════════════════════════════════════════════════════════════╝

MODEL_NAME = 'yolo11l.pt'     # Mejor balance actual para precisión/costo en H100
# MODEL_NAME = 'yolo11x.pt'   # Más preciso en algunos casos, pero más caro
# MODEL_NAME = 'yolo26l.pt'   # Alternativa a comparar luego si quieres benchmarks
# MODEL_NAME = 'yolo26x.pt'   # Más costoso

# Cargar modelo con pesos COCO pre-entrenados
model = YOLO(MODEL_NAME)

model_tag = MODEL_NAME.replace('.pt', '')
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
experiment_name = f'v5-h100-{model_tag}_{timestamp}'

is_yolo26 = 'yolo26' in MODEL_NAME
if is_yolo26:
    print("🚀 YOLO26 detectado → arquitectura más nueva")

print(f"\n🧠 Modelo: {MODEL_NAME}")
print(f"📝 Experimento: {experiment_name}")
print(f"🖥️ GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB)")

info = model.info()
print(f"\n📊 Arquitectura:")
print(f"   Parámetros: {info[1]/1e6:.1f}M")
print(f"   GFLOPs:     {info[2]:.1f}")
print(f"   Capas:      {info[0]}")



🧠 Modelo: yolo11l.pt
📝 Experimento: v4-pretrained-yolo11l_20260307_021323
🖥️ GPU: NVIDIA H100 80GB HBM3 (79 GB)
YOLO11l summary: 357 layers, 25,372,160 parameters, 0 gradients, 87.6 GFLOPs

📊 Arquitectura:
   Parámetros: 25.4M
   GFLOPs:     0.0
   Capas:      357


---
## ⚙️ 4. Auto-Batch (Encontrar batch óptimo)

In [19]:
import os
import psutil

# ╔════════════════════════════════════════════════════════════╗
# ║  PERFIL DE ENTRENAMIENTO                                   ║
# ╚════════════════════════════════════════════════════════════╝

TRAIN_PROFILE = 'balanced'   # 'quick' | 'balanced' | 'final'

PROFILE_CONFIGS = {
    'quick': {
        'imgsz': 1024,
        'batch': 20,
        'workers': 12,
        'cache': False,
        'deterministic': False,
        'plots': False,
        'save_period': 50,
        'epochs': 50,
        'patience_frac': 0.20,
        'label': 'pruebas rápidas y tuning barato'
    },
    'balanced': {
        'imgsz': 1280,
        'batch': 16,
        'workers': 12,
        'cache': False,
        'deterministic': False,
        'plots': False,
        'save_period': 25,
        'epochs': 100,
        'patience_frac': 0.20,
        'label': 'mejor balance entre precisión y costo'
    },
    'final': {
        'imgsz': 1280,
        'batch': 16,
        'workers': 8,
        'cache': False,
        'deterministic': True,
        'plots': True,
        'save_period': 10,
        'epochs': 120,
        'patience_frac': 0.25,
        'label': 'corrida final reproducible'
    }
}

assert TRAIN_PROFILE in PROFILE_CONFIGS, f"Perfil inválido: {TRAIN_PROFILE}"
profile = PROFILE_CONFIGS[TRAIN_PROFILE]

gpu_name = torch.cuda.get_device_name(0)
ram_total_gb = psutil.virtual_memory().total / (1024**3)
ram_free_gb = psutil.virtual_memory().available / (1024**3)

IMGSZ = profile['imgsz']
BATCH_SIZE = profile['batch']
WORKERS = min(profile['workers'], os.cpu_count() or profile['workers'])
CACHE_STRATEGY = profile['cache']
DETERMINISTIC = profile['deterministic']
PLOTS = profile['plots']
SAVE_PERIOD = profile['save_period']
DEFAULT_EPOCHS = profile['epochs']
PATIENCE_FRAC = profile['patience_frac']

# Fallback simple por si cambias de GPU
if 'H100' not in gpu_name:
    if BATCH_SIZE > 12:
        BATCH_SIZE = 12
    WORKERS = min(WORKERS, 8)

print(f"⚙️ Perfil seleccionado: {TRAIN_PROFILE} → {profile['label']}")
print(f"🖥️ GPU detectada: {gpu_name}")
print(f"💾 RAM libre: {ram_free_gb:.1f} / {ram_total_gb:.1f} GB")

print(f"\n📊 Configuración optimizada:")
print(f"   ImgSz:           {IMGSZ}")
print(f"   Batch:           {BATCH_SIZE}")
print(f"   Workers:         {WORKERS}")
print(f"   Cache:           {CACHE_STRATEGY}")
print(f"   Deterministic:   {DETERMINISTIC}")
print(f"   Plots:           {PLOTS}")
print(f"   Save period:     {SAVE_PERIOD}")
print(f"   Epochs default:  {DEFAULT_EPOCHS}")


⚙️ Calculando batch size óptimo para imgsz=1280...
   (esto toma ~30 segundos)

AutoBatch: Computing optimal batch size for imgsz=1280 at 60.0% CUDA memory utilization.
WARNING ⚠️ AutoBatch: intended for CUDA devices, using default batch-size 16

📊 Resultados (imgsz=1280):
   Batch máximo detectado: 16
   Batch final (seguro):    12
   Cache strategy:          False

📋 Referencia por GPU (YOLO11l @ 1280):
   T4  (15GB): batch 2-4
   L4  (24GB): batch 4-8
   A100(40GB): batch 8-12
   A100(80GB): batch 12-18 ⭐
   H100(80GB): batch 18-24 🚀

✅ Batch final: 12


---
## 🚀 5. Hiperparámetros y Entrenamiento

In [20]:
# ╔════════════════════════════════════════════════════════════╗
# ║             HIPERPARÁMETROS - AJUSTAR AQUÍ                  ║
# ╚════════════════════════════════════════════════════════════╝

EPOCHS = DEFAULT_EPOCHS      # quick=50 | balanced=100 | final=120
CONF_THRESHOLD = 0.001       # Validación (mAP completo)
CONF_INFERENCE = 0.30        # Producción
PATIENCE = max(10, int(EPOCHS * PATIENCE_FRAC))

# Para BATCH_SIZE <= 4 podrías usar acumulación; aquí no hace falta
ACCUMULATE = 1

print(f"\n{'='*70}")
print(f"🚀 CONFIGURACIÓN v5 - H100 Optimizada")
print(f"{'='*70}")
print(f"\n📋 Hiperparámetros:")
print(f"   Perfil:         {TRAIN_PROFILE}")
print(f"   Modelo:         {MODEL_NAME}")
print(f"   Epochs:         {EPOCHS}")
print(f"   Batch:          {BATCH_SIZE}")
print(f"   Resolución:     {IMGSZ}x{IMGSZ}")
print(f"   Optimizer:      SGD (manual, no auto)")
print(f"   LR:             0.01 → 0.0015 (LRF=0.15, cosine)")
print(f"   Cache:          {CACHE_STRATEGY}")
print(f"   Workers:        {WORKERS}")
print(f"   Plots:          {PLOTS}")
print(f"   Save period:    {SAVE_PERIOD}")
print(f"\n🎯 Thresholds:")
print(f"   Conf (val):     {CONF_THRESHOLD}")
print(f"   Conf (prod):    {CONF_INFERENCE}")
print(f"   IOU (train):    0.7")
print(f"   Patience:       {PATIENCE} epochs")
print(f"{'='*70}")



🚀 CONFIGURACIÓN v4 - YOLO11l Pre-entrenado COCO

📋 Hiperparámetros:
   Modelo:         yolo11l.pt
   Epochs:         100 (PRUEBA)
                   (Aumentar a 250 para entrenamiento final)
   Batch:          12 (effective: 12)
   Resolución:     1280x1280
   Optimizer:      auto
   LR:             0.01 → 0.0015 (LRF=0.15, cosine)
   Cache:          False

🎯 Thresholds:
   Conf (val):     0.001
   Conf (prod):    0.3
   IOU (train):    0.7
   Patience:       30 epochs


### 🎮 Control: Pause y Resume

Puedes pausar el entrenamiento en cualquier momento:
- **Detener Colab**: Runtime → Interrupt execution
- **Reanudar**: Cambia `RESUME_TRAINING = True` y re-ejecuta

In [21]:
# ╔════════════════════════════════════════════════════════════╗
# ║  CONTROL DE PAUSE/RESUME                                   ║
# ╚════════════════════════════════════════════════════════════╝

RESUME_TRAINING = False  # Cambiar a True para reanudar

checkpoint_path = None
if RESUME_TRAINING:
    import glob
    checkpoints = sorted(glob.glob(f'/content/drive/MyDrive/SIRCCD_Models/{experiment_name}/weights/last.pt'))
    if not checkpoints:
        checkpoints = sorted(glob.glob('/content/drive/MyDrive/SIRCCD_Models/v5-h100-*/weights/last.pt'))

    if checkpoints:
        checkpoint_path = checkpoints[-1]
        experiment_name = checkpoint_path.split('/')[-3]
        print(f"✅ MODO RESUME ACTIVADO")
        print(f"📍 Checkpoint: {checkpoint_path}")
        print(f"📝 Experimento: {experiment_name}")
        model = YOLO(checkpoint_path)
    else:
        print(f"❌ No se encontraron checkpoints. Iniciando desde cero.")
        RESUME_TRAINING = False
else:
    print(f"▶️  MODO NUEVO ENTRENAMIENTO (pesos COCO pre-entrenados)")
    print(f"\n💡 Para reanudar: RESUME_TRAINING = True")


▶️  MODO NUEVO ENTRENAMIENTO (pesos COCO pre-entrenados)

💡 Para reanudar: RESUME_TRAINING = True


In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║             ENTRENAMIENTO v5                                ║
# ╚════════════════════════════════════════════════════════════╝

print(f"\n{'='*70}")
if RESUME_TRAINING:
    print(f"🔄 REANUDANDO ENTRENAMIENTO")
else:
    print(f"🚀 INICIANDO ENTRENAMIENTO v5 - {MODEL_NAME} ({TRAIN_PROFILE})")
print(f"{'='*70}\n")

results = model.train(
    # === Dataset ===
    data=f'{EXTRACT_DIR}/data.yaml',
    imgsz=IMGSZ,

    # === Training ===
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    resume=RESUME_TRAINING,

    # === Optimizer (manual para control real) ===
    optimizer='SGD',
    lr0=0.01,
    lrf=0.15,
    momentum=0.937,
    cos_lr=True,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    weight_decay=0.0005,

    # === Augmentation (moderada para no disparar costo ni ruido) ===
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=8.0,
    translate=0.08,
    scale=0.4,
    shear=2.0,
    perspective=0.0002,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.05,
    copy_paste=0.10,
    erasing=0.2,
    close_mosaic=10,
    dropout=0.0,

    # === Thresholds ===
    iou=0.7,

    # === Guardado ===
    project='/content/drive/MyDrive/SIRCCD_Models',
    name=experiment_name,
    save_period=SAVE_PERIOD,
    patience=PATIENCE,

    # === Validación ===
    val=True,
    plots=PLOTS,

    # === GPU/Performance ===
    device=0,
    amp=True,
    cache=CACHE_STRATEGY,
    workers=WORKERS,
    deterministic=DETERMINISTIC,
    seed=42,

    # === Transfer Learning ===
    pretrained=True,
    verbose=True,
    rect=False,
)

print(f"\n{'='*70}")
print(f"✅ ENTRENAMIENTO COMPLETADO - {MODEL_NAME} @ {IMGSZ}")
print(f"{'='*70}")
print(f"\n📁 Resultados en: SIRCCD_Models/{experiment_name}/")
print(f"   weights/best.pt  (mejor mAP50-95)")
print(f"   weights/last.pt  (para continuar)")



🚀 INICIANDO ENTRENAMIENTO v4 - yolo11l.pt (COCO pre-entrenado)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sirccd_dataset/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.3, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.15, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v4-pretrained-yolo11l_20260307_0

---
## 🔍 6. Evaluación

In [ ]:
# Evaluación en validación
print("📊 Evaluando en validación...")
val_metrics = best_model.val(
    data=f'{EXTRACT_DIR}/data.yaml',
    split='val',
    conf=CONF_THRESHOLD,
    iou=0.6
)

print(f"\n🎯 Validación:")
print(f"   mAP50:     {val_metrics.box.map50:.4f}")
print(f"   mAP50-95:  {val_metrics.box.map:.4f}")
print(f"   Precision: {val_metrics.box.mp:.4f}")
print(f"   Recall:    {val_metrics.box.mr:.4f}")

In [ ]:
# Evaluación en test
print("📊 Evaluando en test...")
test_metrics = best_model.val(
    data=f'{EXTRACT_DIR}/data.yaml',
    split='test',
    conf=CONF_THRESHOLD,
    iou=0.6
)

print(f"\n🎯 Test:")
print(f"   mAP50:     {test_metrics.box.map50:.4f}")
print(f"   mAP50-95:  {test_metrics.box.map:.4f}")
print(f"   Precision: {test_metrics.box.mp:.4f}")
print(f"   Recall:    {test_metrics.box.mr:.4f}")

In [ ]:
# === COMPARACIÓN CON v1 ===
print("\n" + "="*70)
print("📈 COMPARACIÓN: v1 (YOLOv8m) vs v4 (YOLO11l pre-entrenado)")
print("="*70)

v1 = {'mAP50': 0.73388, 'mAP50-95': 0.45006, 'Precision': 0.73254, 'Recall': 0.68304}
v4_val = {
    'mAP50': val_metrics.box.map50, 'mAP50-95': val_metrics.box.map,
    'Precision': val_metrics.box.mp, 'Recall': val_metrics.box.mr
}
v4_test = {
    'mAP50': test_metrics.box.map50, 'mAP50-95': test_metrics.box.map,
    'Precision': test_metrics.box.mp, 'Recall': test_metrics.box.mr
}

print(f"\n{'Métrica':<12} {'v1 (YOLOv8m)':>14} {'v4 (val)':>10} {'v4 (test)':>10} {'Δ val':>8} {'Δ%':>8}")
print(f"{'-'*66}")
for key in v1:
    delta = v4_val[key] - v1[key]
    delta_pct = (delta / v1[key]) * 100
    sign = '+' if delta >= 0 else ''
    print(f"{key:<12} {v1[key]:>14.4f} {v4_val[key]:>10.4f} {v4_test[key]:>10.4f} "
          f"{sign}{delta:>7.4f} {sign}{delta_pct:>6.1f}%")

# Por clase
print(f"\n📊 Métricas por Clase (test):")
print(f"{'Clase':<10} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
print(f"{'-'*50}")
for i, name in enumerate(['bache', 'grieta']):
    print(f"{name:<10} {test_metrics.box.p[i]:>10.3f} {test_metrics.box.r[i]:>10.3f} "
          f"{test_metrics.box.ap50[i]:>10.3f} {test_metrics.box.ap[i]:>10.3f}")

In [ ]:
# === TEST-TIME AUGMENTATION (TTA) ===
print("\n🔬 Evaluación con TTA (Test-Time Augmentation):")
tta_metrics = best_model.val(
    data=f'{EXTRACT_DIR}/data.yaml',
    split='test',
    augment=True,
    conf=CONF_THRESHOLD,
    iou=0.6
)

print(f"\n📊 Comparación (test):")
print(f"   {'':>12} {'Sin TTA':>10} {'Con TTA':>10} {'Δ':>8} {'Δ%':>8}")
print(f"   {'-'*48}")
for label, no_tta, tta in [
    ('mAP50', test_metrics.box.map50, tta_metrics.box.map50),
    ('mAP50-95', test_metrics.box.map, tta_metrics.box.map),
    ('Precision', test_metrics.box.mp, tta_metrics.box.mp),
    ('Recall', test_metrics.box.mr, tta_metrics.box.mr),
]:
    d = tta - no_tta
    d_pct = (d / no_tta) * 100 if no_tta > 0 else 0
    s = '+' if d >= 0 else ''
    print(f"   {label:<12} {no_tta:>10.4f} {tta:>10.4f} {s}{d:>7.4f} {s}{d_pct:>6.2f}%")

---
## 📈 7. Visualizaciones

In [ ]:
from IPython.display import Image, display

results_dir = f'/content/drive/MyDrive/SIRCCD_Models/{experiment_name}'

for filename, title in [
    ('results.png', '📈 Curvas de Entrenamiento'),
    ('confusion_matrix_normalized.png', '🔢 Matriz de Confusión'),
    ('F1_curve.png', '📊 Curva F1'),
    ('PR_curve.png', '📊 Precision-Recall'),
]:
    path = os.path.join(results_dir, filename)
    if os.path.exists(path):
        print(f"\n{title}:")
        display(Image(filename=path, width=800))

In [ ]:
import random

# Predicciones en test aleatorio
test_images = glob.glob(f'{EXTRACT_DIR}/images/test/*.jpg')
sample = random.sample(test_images, min(12, len(test_images)))

print(f"🔍 Predicciones de ejemplo (conf > {CONF_INFERENCE}):")

results = best_model.predict(
    source=sample,
    conf=CONF_INFERENCE,
    iou=0.5,
    save=True,
    project='/content/predictions_v4',
    name='test',
    exist_ok=True
)

total_dets = 0
for r in results:
    fname = os.path.basename(r.path)
    num = len(r.boxes)
    total_dets += num
    if num > 0:
        det = ', '.join(f"{r.names[int(c)]}({float(conf):.2f})"
                       for c, conf in zip(r.boxes.cls, r.boxes.conf))
        print(f"  ✅ {fname}: {det}")
    else:
        print(f"  ⬜ {fname}: Sin detecciones")

print(f"\n📊 Resumen: {total_dets} detecciones en {len(sample)} imágenes")
print(f"   Promedio: {total_dets/len(sample):.1f} detecciones/imagen")

# Mostrar algunas
pred_imgs = sorted(glob.glob('/content/predictions_v4/test/*.jpg'))[:6]
for p in pred_imgs:
    display(Image(filename=p, width=500))

---
## 💾 8. Exportar para Producción

In [ ]:
import shutil

export_dir = f'/content/drive/MyDrive/SIRCCD_Models/{experiment_name}/exports'
os.makedirs(export_dir, exist_ok=True)

# PyTorch
shutil.copy(best_path, f'{export_dir}/best.pt')
print(f"✅ PyTorch: {export_dir}/best.pt")

# ONNX
onnx_path = best_model.export(format='onnx', imgsz=IMGSZ, simplify=True, opset=17)
shutil.copy(onnx_path, f'{export_dir}/best.onnx')
print(f"✅ ONNX: {export_dir}/best.onnx")

# TorchScript
ts_path = best_model.export(format='torchscript', imgsz=IMGSZ)
shutil.copy(ts_path, f'{export_dir}/best.torchscript')
print(f"✅ TorchScript: {export_dir}/best.torchscript")

# Tamaños
print(f"\n📊 Tamaños:")
for f in os.listdir(export_dir):
    size = os.path.getsize(os.path.join(export_dir, f)) / (1024*1024)
    print(f"   {f}: {size:.1f} MB")

---
## 📝 9. Resumen Final

In [ ]:
import json
from datetime import datetime

summary = {
    'version': 'v5-h100-optimized',
    'fecha': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'estrategia': f'Transfer learning {MODEL_NAME} optimizado para H100 ({TRAIN_PROFILE})',
    'modelo': MODEL_NAME,
    'perfil': TRAIN_PROFILE,
    'parametros': f'{info[1]/1e6:.1f}M',
    'gflops': f'{info[2]:.1f}',
    'clases': ['bache', 'grieta'],
    'configuracion': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'imgsz': IMGSZ,
        'optimizer': 'SGD',
        'lr0': 0.01,
        'lrf': 0.15,
        'cos_lr': True,
        'patience': PATIENCE,
        'workers': WORKERS,
        'cache': CACHE_STRATEGY,
        'deterministic': DETERMINISTIC,
        'plots': PLOTS,
        'conf_threshold': CONF_THRESHOLD,
        'conf_inference': CONF_INFERENCE,
        'iou_threshold': 0.7
    },
    'gpu': torch.cuda.get_device_name(0),
    'metricas': {
        'v1_baseline': {'mAP50': 0.73388, 'mAP50-95': 0.45006, 'P': 0.73254, 'R': 0.68304},
        'v5_val': {
            'mAP50': float(val_metrics.box.map50),
            'mAP50-95': float(val_metrics.box.map),
            'P': float(val_metrics.box.mp),
            'R': float(val_metrics.box.mr)
        },
        'v5_test': {
            'mAP50': float(test_metrics.box.map50),
            'mAP50-95': float(test_metrics.box.map),
            'P': float(test_metrics.box.mp),
            'R': float(test_metrics.box.mr)
        },
        'v5_test_tta': {
            'mAP50': float(tta_metrics.box.map50),
            'mAP50-95': float(tta_metrics.box.map),
            'P': float(tta_metrics.box.mp),
            'R': float(tta_metrics.box.mr)
        }
    }
}

summary_path = f'/content/drive/MyDrive/SIRCCD_Models/{experiment_name}/training_summary_v5.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n" + "="*70)
print("🎉 ENTRENAMIENTO v5 - RESUMEN FINAL")
print("="*70)
print(f"\n🧠 Modelo:  {MODEL_NAME}")
print(f"📊 Params:  {info[1]/1e6:.1f}M")
print(f"🏷️ Perfil:  {TRAIN_PROFILE}")
print(f"🔄 Epochs:  {EPOCHS}")
print(f"📐 ImgSz:   {IMGSZ}")
print(f"🖥️ GPU:     {torch.cuda.get_device_name(0)}")
print(f"🎯 Clases:  bache, grieta")

print(f"\n{'Métrica':<12} {'v1':>10} {'v5 val':>10} {'v5 test':>10} {'v5+TTA':>10} {'Δ%':>8}")
print(f"{'-'*63}")
for key, k1, k5 in [
    ('mAP50', 'mAP50', 'mAP50'),
    ('mAP50-95', 'mAP50-95', 'mAP50-95'),
    ('Precision', 'P', 'P'),
    ('Recall', 'R', 'R')
]:
    v1_val = summary['metricas']['v1_baseline'][k1]
    v5_test_val = summary['metricas']['v5_test'][k5]
    delta_pct = ((v5_test_val - v1_val) / v1_val) * 100
    sign = '+' if delta_pct >= 0 else ''

    print(f"{key:<12} {v1_val:>10.4f} "
          f"{summary['metricas']['v5_val'][k5]:>10.4f} "
          f"{v5_test_val:>10.4f} "
          f"{summary['metricas']['v5_test_tta'][k5]:>10.4f} "
          f"{sign}{delta_pct:>6.1f}%")

print(f"\n💾 Guardado en: SIRCCD_Models/{experiment_name}/")
print(f"📄 Resumen JSON: training_summary_v5.json")
print("="*70)


---
## 🔮 10. ¿Qué sigue ahora?

### Si quieres gastar menos mientras pruebas
Usa:
```python
TRAIN_PROFILE = 'quick'
```
Eso te da una corrida más barata para probar cambios de dataset, augmentations o hiperparámetros.

### Si quieres el mejor balance real en la H100
Usa:
```python
TRAIN_PROFILE = 'balanced'
```
Ese es el perfil recomendado para este proyecto con datasets de detección.

### Si ya encontraste una configuración ganadora
Usa:
```python
TRAIN_PROFILE = 'final'
RESUME_TRAINING = False
```
Solo para la corrida final, cuando ya no estés experimentando tanto.

### Próximos pasos recomendados
1. Comparar `quick` vs `balanced` en métricas reales.
2. Revisar las imágenes con labels duplicadas o dudosas.
3. Evaluar errores típicos:
   - grietas finas no detectadas,
   - baches confundidos con sombras,
   - cajas muy grandes o muy pequeñas.
4. Guardar el mejor `best.pt` y moverlo al backend.
5. Luego, si quieres más precisión visual en contornos, preparar un subconjunto para segmentación.
